In [5]:
import os
import psycopg2
import pandas as pd
import warnings

warnings.filterwarnings("ignore")


file_path = "cleaned_health_data.csv"
health_df = pd.read_csv(file_path)


health_df = health_df.loc[:, ~health_df.columns.str.contains('^Unnamed')]


health_df.columns = health_df.columns.str.strip()


health_df.columns = [col.replace(" ", "_").replace("(", "").replace(")", "").replace("%", "pct") for col in health_df.columns]


def infer_sql_type(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    else:
        return "TEXT"


conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="root",
    host="localhost",
    port="5432"
)
cur = conn.cursor()


table_name = "smartwatch_data"
columns = health_df.dtypes
sql_columns = ",\n  ".join([f'"{col}" {infer_sql_type(dtype)}' for col, dtype in columns.items()])

create_stmt = f"""
CREATE TABLE "{table_name}" (
  {sql_columns}
);
"""

cur.execute(f'DROP TABLE IF EXISTS "{table_name}" CASCADE;')
cur.execute(create_stmt)
conn.commit()
print("Table recreated with schema:")
print(create_stmt)


columns_list = list(health_df.columns)
placeholders = ', '.join(['%s'] * len(columns_list))
quoted_cols = ', '.join([f'"{col}"' for col in columns_list])

insert_stmt = f'INSERT INTO "{table_name}" ({quoted_cols}) VALUES ({placeholders})'

for _, row in health_df.iterrows():
    row_values = [None if pd.isna(val) else val for val in row[columns_list]]
    cur.execute(insert_stmt, tuple(row_values))

conn.commit()
print("Data inserted successfully")


df_from_db = pd.read_sql(f'SELECT * FROM "{table_name}"', conn)
cur.close()
conn.close()

print("Data fetched from DB:")
print(df_from_db.head())


Table recreated with schema:

CREATE TABLE "smartwatch_data" (
  "user_id" INT,
  "heart_rate_bpm" FLOAT,
  "blood_oxygen_level_pct" FLOAT,
  "step_count" FLOAT,
  "sleep_duration_hours" FLOAT,
  "activity_level" TEXT,
  "stress_level" TEXT
);

Data inserted successfully
Data fetched from DB:
   user_id  heart_rate_bpm  blood_oxygen_level_pct    step_count  \
0     4174       58.939776               98.809650   5450.390578   
1     1860      247.803052               97.052954   2826.521994   
2     2294       40.000000               96.894213  13797.338044   
3     2130       61.950165               98.583797  15679.067648   
4     2095       96.285938               94.202910  10205.992256   

   sleep_duration_hours activity_level stress_level  
0              7.167236  Highly active          Low  
1              6.501197  Highly active     Moderate  
2              7.367790         Active          Low  
3              6.501197  Highly active     Moderate  
4              8.378343  Hi